Реализация адаптивных и стохастических методов $AdaGrad, AdamW$.
Подходы к борьбе с переобучением (отложенная выборка, кросс-валидация, регуляризации L1, L2):
- Отложенная выборка (Hold-out Validation): Техника однократного разделения исходного множества на изолированные подмножества (например, Train и Test). Если тестовое подмножество используется исключительно для финальной фиксации метрики, процедура корректна. Если же по отложенной выборке подбираются гиперпараметры ($\lambda$) или принимается решение о досрочной остановке (Early Stopping), для предотвращения утечки данных (data leakage) требуется трехкомпонентное разбиение: Train (обучение), Validation (настройка $\lambda$, Early Stopping) и Test (финальный аудит).
- Кросс-валидация (K-Fold Cross-Validation): Циклическое разбиение данных на $K$ равных блоков (фолдов). На каждой из $K$ итераций модель обучается на $K-1$ блоках, а тестируется на оставшемся одном. Метод минимизирует дисперсию оценки качества модели при малом объеме выборки.

Для сопоставимости силы штрафа при изменении объема выборки $N$, регуляризационные члены нормируются на размер данных:
L1-регуляризация (аппроксимация Lasso подградиентным методом):
$$ J_{L1} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}i)^2 + \frac{\lambda}{N} \sum{j=1}^M |w_j| $$
L2-регуляризация (Ridge-регрессия):
$$ J_{L2} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}i)^2 + \frac{\lambda}{N} \sum{j=1}^M w_j^2 $$
Где $N$ — количество объектов, $M$ — количество признаков, а вектор весов $w$ не включает в себя свободный член (bias).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
target_relative_path = Path("..") / "Dataset (Farhat)" / "dataset_sample_1000.csv"
dataset_path = target_relative_path.resolve()

if not dataset_path.exists():
    print(f"Критическая ошибка: Файл не найден.")
    print(f"Ожидаемый абсолютный путь: {dataset_path}")
    print("Проверьте корректность имени папки")
    sys.exit(1)
print(f"Успешно обнаружен файл датасета по пути: {dataset_path}")

In [ ]:
torch.manual_seed(67)
np.random.seed(67)

df = pd.read_csv(dataset_path)
X_raw = df.drop(columns=['Цена_log']).values
y_raw = df['Цена_log'].values.reshape(-1, 1)

# Разбиение на Train и Test (Hold-out validation)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=67
)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_test_scaled = scaler_X.transform(X_test_raw)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_raw)
y_test_scaled = scaler_y.transform(y_test_raw)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# DataLoader для стохастического градиентного спуска (Mini-Batch SGD)
BATCH_SIZE = 64
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Сетка параметров
lambdas = np.logspace(-3, 1, num=5)
EPOCHS = 200
INPUT_DIM = X_train_tensor.shape[1]

# Структура для логирования кривых обучения
history_data = {}

## *AdaGrad* (Adaptive Gradient Algorithm)
- Суть: Алгоритм адаптирует темп обучения покоординатно, масштабируя текущий градиент обратно пропорционально корню из суммы квадратов всех исторических градиентов данного параметра. Это позволяет делать большие шаги по редким (разреженным) признакам и уменьшать шаг по часто обновляемым.

Математический шаг оптимизации:
Пусть $g_t = \nabla_{\theta} L(\theta_t)$ — градиент функции потерь по весам $\theta$ на шаге $t$. Аккумуляция квадратов градиентов $G_t \in \mathbb{R}^d$ и обновление параметров выполняются покоординатно (оператор $\odot$ обозначает умножение Адамара):
$$ G_t = G_{t-1} + g_t \odot g_t $$
$$ \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t} + \epsilon} \odot g_t $$
Где $\eta$ — базовый темп обучения, а $\epsilon > 0$ — сглаживающий параметр, предотвращающий деление на ноль (в реализации torch.optim.Adagrad по умолчанию $\epsilon = 10^{-10}$). Монотонное возрастание компонент вектора $G_t$ гарантирует монотонное убывание эффективного шага.

## *AdamW* (Adam with Decoupled Weight Decay) 
- Суть: Adam комбинирует идеи метода накопления импульса (Momentum) и покоординатного масштабирования шага (RMSProp). Алгоритм вычисляет экспоненциально затухающие средние прошлых градиентов ($m_t$) и квадратов прошлых градиентов ($v_t$).
 
Математический шаг оптимизации:
Вычисление смещенных оценок первого и второго моментов градиента:
$$ m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t $$
$$ v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t \odot g_t $$

Поскольку $m_0$ и $v_0$ инициализируются нулями, оценки смещены в сторону нуля (особенно на первых итерациях при малых значениях $\beta_1$, $\beta_2$). Для компенсации вычисляются несмещенные оценки моментов:
$$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} $$

Финальный шаг обновления параметров:
$$ \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \odot \hat{m}_t $$

Где гиперпараметры по умолчанию имеют канонические значения: $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$ .При использовании weight_decay в классическом Adam, градиент L2-штрафа прибавляется напрямую к градиенту потерь до расчета моментов

In [ ]:
configurations = [
    {"opt": "AdaGrad", "reg": "L1", "lr": 0.05},
    {"opt": "AdaGrad", "reg": "L2", "lr": 0.05},
    {"opt": "Adam",    "reg": "L1", "lr": 0.01},
    {"opt": "Adam",    "reg": "L2", "lr": 0.01}
]

for config in configurations:
    opt_name = config["opt"]
    reg_type = config["reg"]
    learning_rate = config["lr"]
    
    for lmbda in lambdas:
        # Инициализируем воспроизводимое состояние сети
        torch.manual_seed(67)
        model = nn.Linear(INPUT_DIM, 1)
        criterion = nn.MSELoss()
        
        wd_param = lmbda if reg_type == "L2" else 0.0
        if opt_name == "AdaGrad":
            optimizer = optim.Adagrad(model.parameters(), lr=learning_rate, weight_decay=wd_param)
        elif opt_name == "Adam":
            optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=wd_param)
            
        train_epochs_loss = []
        test_epochs_loss = []
        # Стохастический мини-батч цикл обучения
        for epoch in range(EPOCHS):
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                predictions = model(batch_X)
                loss = criterion(predictions, batch_y)
                if reg_type == "L1":
                    l1_penalty = sum(p.abs().sum() for name, p in model.named_parameters() if 'bias' not in name)
                    loss = loss + lmbda * l1_penalty
                loss.backward()
                optimizer.step()
    
            model.eval()
            with torch.no_grad():
                current_train_mse = criterion(model(X_train_tensor), y_train_tensor).item()
                current_test_mse = criterion(model(X_test_tensor), y_test_tensor).item()
            train_epochs_loss.append(current_train_mse)
            test_epochs_loss.append(current_test_mse)

        exp_key = f"{opt_name}_{reg_type}_lambda_{lmbda:.4f}"
        history_data[exp_key] = {
            "train": train_epochs_loss,
            "test": test_epochs_loss,
            "final_test": test_epochs_loss[-1]
        }

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, config in enumerate(configurations):
    ax = axes[i]
    opt_name = config["opt"]
    reg_type = config["reg"]
    ax.set_title(f"Динамика потерь: {opt_name} с регуляризацией {reg_type}", fontsize=12, fontweight='bold')
    for i, lmbda in enumerate(lambdas):
        exp_key = f"{opt_name}_{reg_type}_lambda_{lmbda:.4f}"
        curves = history_data[exp_key]
        ax.plot(curves["test"], color=colors[i], linestyle='-', linewidth=1.8, label=f'Test λ = {lmbda:.3f}')
        ax.plot(curves["train"], color=colors[i], linestyle='--', alpha=0.5, linewidth=1.0)
    ax.set_xlabel("Эпохи")
    ax.set_ylabel("MSE Loss (Стандартизированный)")
    ax.set_yscale('log')
    ax.grid(True, which="both", linestyle=':', alpha=0.6)
    ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

print("\n" + "~" * 20 + " РЕЙТИНГ МОДЕЛЕЙ (ПО ТЕСТОВОМУ MSE) " + "~" * 20)
sorted_results = sorted(history_data.items(), key=lambda item: item[1]["final_test"])
for name, data in sorted_results[:10]:
    print(f"Конфигурация: {name:<35} | Финальный Test MSE: {data['final_test']:.5f}")
    